# WTI Flat Price Contracts — Open Interest EDA

Analyze the relative size of each WTI flat price contract code from `disaggregated_combined.csv`.

Flat price codes (from `docs/wti_cot_mapping.md`):
- `067651` — CRUDE OIL, LIGHT SWEET - NEW YORK MERCANTILE EXCHANGE (physical WTI)
- `067411` — CRUDE OIL, LIGHT SWEET - ICE FUTURES EUROPE (CFTC-reported)
- `06765I` — WTI CRUDE OIL FINANCIAL - NEW YORK MERCANTILE EXCHANGE
- `067A55` — Micro WTI CRUDE OIL FINANCIAL - NEW YORK MERCANTILE EXCHANGE
- `067655` — E-MINI CRUDE OIL, LIGHT SWEET - NEW YORK MERCANTILE EXCHANGE
- `06765C` — CRUDE OIL AVG PRICE OPTIONS - NEW YORK MERCANTILE EXCHANGE
- `06765B` — EUR STYLE CRUDE OIL OPTIONS - NEW YORK MERCANTILE EXCHANGE
- `06741Q` — WTI CRUDE OIL 1ST LINE - ICE FUTURES ENERGY DIV
- `06765A` — WTI CRUDE OIL CALENDAR SWAP - NEW YORK MERCANTILE EXCHANGE

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
df = pd.read_csv("../../downloads/cftc/disaggregated_combined.csv", low_memory=False)
print(f"Total rows: {len(df):,}")
df.head(2)

Total rows: 184,455


,id,market_and_exchange_names,report_date_as_yyyy_mm_dd,yyyy_report_week_ww,contract_market_name,cftc_contract_market_code,cftc_market_code,cftc_region_code,cftc_commodity_code,commodity_name,...,change_in_m_money_long_all,change_in_m_money_short_all,change_in_m_money_spread,change_in_other_rept_long,change_in_other_rept_short,change_in_other_rept_spread,change_in_tot_rept_long_all,change_in_tot_rept_short,change_in_nonrept_long_all,change_in_nonrept_short_all
0,060613001602C,WHEAT - CHICAGO BOARD OF TRADE,2006-06-13T00:00:00.000,2006 Report Week 24,WHEAT-SRW,001602,CBT,CHI,1,WHEAT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,060613001612C,WHEAT - KANSAS CITY BOARD OF TRADE,2006-06-13T00:00:00.000,2006 Report Week 24,WHEAT-HRW,001612,KCBT,CHI,1,WHEAT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
WTI_FLAT_PRICE_CODES = [
    "067651",  # CRUDE OIL, LIGHT SWEET - NYMEX (physical)
    "067411",  # CRUDE OIL, LIGHT SWEET - ICE FUTURES EUROPE
    "06765I",  # WTI CRUDE OIL FINANCIAL - NYMEX
    "067A55",  # Micro WTI CRUDE OIL FINANCIAL - NYMEX
    "067655",  # E-MINI CRUDE OIL, LIGHT SWEET - NYMEX
    "06765C",  # CRUDE OIL AVG PRICE OPTIONS - NYMEX
    "06765B",  # EUR STYLE CRUDE OIL OPTIONS - NYMEX
    "06741Q",  # WTI CRUDE OIL 1ST LINE - ICE FUTURES ENERGY DIV
    "06765A",  # WTI CRUDE OIL CALENDAR SWAP - NYMEX (added per Marouen)
]

wti = df[df["cftc_contract_market_code"].isin(WTI_FLAT_PRICE_CODES)].copy()
wti["report_date_as_yyyy_mm_dd"] = pd.to_datetime(wti["report_date_as_yyyy_mm_dd"])
wti["open_interest_all"] = pd.to_numeric(wti["open_interest_all"], errors="coerce")
print(f"WTI flat price rows: {len(wti):,}")
print(f"Date range: {wti['report_date_as_yyyy_mm_dd'].min().date()} to {wti['report_date_as_yyyy_mm_dd'].max().date()}")
print(f"Codes found: {sorted(wti['cftc_contract_market_code'].unique())}")

## Average Open Interest by Contract Code

In [4]:
# Build a readable label for each code
code_labels = (
    wti.groupby("cftc_contract_market_code")["market_and_exchange_names"]
    .first()
    .to_dict()
)

avg_oi = (
    wti.groupby("cftc_contract_market_code")["open_interest_all"]
    .mean()
    .sort_values(ascending=False)
    .to_frame("avg_open_interest")
)
avg_oi["contract_name"] = avg_oi.index.map(code_labels)
avg_oi["source"] = "CFTC"
avg_oi["pct_of_total"] = (avg_oi["avg_open_interest"] / avg_oi["avg_open_interest"].sum() * 100)
avg_oi["avg_open_interest"] = avg_oi["avg_open_interest"].round(0).astype(int)
avg_oi["pct_of_total"] = avg_oi["pct_of_total"].round(2)

avg_oi[["contract_name", "source", "avg_open_interest", "pct_of_total"]]

,contract_name,source,avg_open_interest,pct_of_total
cftc_contract_market_code,,,,
067651,"CRUDE OIL, LIGHT SWEET - NEW YORK MERCANTILE E...",CFTC,2581988,63.13
067411,"CRUDE OIL, LIGHT SWEET - ICE EUROPE",CFTC,696658,17.03
06765C,CRUDE OIL AVG PRICE OPTIONS - NEW YORK MERCANT...,CFTC,225574,5.52
06765A,WTI CRUDE OIL CALENDAR SWAP - NEW YORK MERCANT...,CFTC,160858,3.93
06765I,WTI CRUDE OIL FINANCIAL - NEW YORK MERCANTILE ...,CFTC,108072,2.64
06765B,EUR STYLE CRUDE OIL OPTIONS - NEW YORK MERCANT...,CFTC,91675,2.24
06741Q,WTI CRUDE OIL 1ST LINE - ICE FUTURES ENERGY DIV,CFTC,86510,2.12
067A55,Micro WTI CRUDE OIL FINANCIAL - NEW YORK MERCA...,CFTC,68564,1.68
06765G,DUBAI CRUDE OIL CALENDAR SWAP - NEW YORK MERCA...,CFTC,35423,0.87


## Last Report Date by Contract Code

In [5]:
last_report = (
    wti.groupby("cftc_contract_market_code")
    .agg(
        contract_name=("market_and_exchange_names", "first"),
        last_report_date=("report_date_as_yyyy_mm_dd", "max"),
        last_report_oi=("open_interest_all", "last"),
    )
    .sort_values("last_report_date", ascending=False)
)
last_report["source"] = "CFTC"
last_report["last_report_date"] = last_report["last_report_date"].dt.date
last_report

,contract_name,last_report_date,last_report_oi,source
cftc_contract_market_code,,,,
067411,"CRUDE OIL, LIGHT SWEET - ICE EUROPE",2026-03-31,1184319,CFTC
06741Q,WTI CRUDE OIL 1ST LINE - ICE FUTURES ENERGY DIV,2026-03-31,38890,CFTC
067651,"CRUDE OIL, LIGHT SWEET - NEW YORK MERCANTILE E...",2026-03-31,3039583,CFTC
06765A,WTI CRUDE OIL CALENDAR SWAP - NEW YORK MERCANT...,2026-03-31,235004,CFTC
06765C,CRUDE OIL AVG PRICE OPTIONS - NEW YORK MERCANT...,2026-03-31,173578,CFTC
067A55,Micro WTI CRUDE OIL FINANCIAL - NEW YORK MERCA...,2026-03-31,76474,CFTC
06765B,EUR STYLE CRUDE OIL OPTIONS - NEW YORK MERCANT...,2022-08-16,40152,CFTC
06765I,WTI CRUDE OIL FINANCIAL - NEW YORK MERCANTILE ...,2013-09-17,48209,CFTC
06765G,DUBAI CRUDE OIL CALENDAR SWAP - NEW YORK MERCA...,2012-10-30,30649,CFTC


## Open Interest Share — Pie Chart

In [6]:
fig = px.pie(
    avg_oi.reset_index(),
    values="avg_open_interest",
    names="cftc_contract_market_code",
    hover_data=["contract_name"],
    title="WTI Flat Price Contracts — Average Open Interest Share",
)
fig.update_traces(textinfo="label+percent", textposition="outside")
fig.show()

## Open Interest Over Time by Contract Code

In [7]:
fig = px.line(
    wti.sort_values("report_date_as_yyyy_mm_dd"),
    x="report_date_as_yyyy_mm_dd",
    y="open_interest_all",
    color="cftc_contract_market_code",
    hover_data=["market_and_exchange_names"],
    title="WTI Flat Price Contracts — Open Interest Over Time",
    labels={
        "report_date_as_yyyy_mm_dd": "Report Date",
        "open_interest_all": "Open Interest",
        "cftc_contract_market_code": "Contract Code",
    },
)
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.3))
fig.show()

## Open Interest Over Time — Stacked Area (Percentage)

In [8]:
# Pivot to get OI per code per date, then compute percentage share
pivot = wti.pivot_table(
    index="report_date_as_yyyy_mm_dd",
    columns="cftc_contract_market_code",
    values="open_interest_all",
    aggfunc="sum",
).fillna(0)

pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig = go.Figure()
for col in pct.columns:
    label = code_labels.get(col, col)
    fig.add_trace(go.Scatter(
        x=pct.index, y=pct[col],
        name=col,
        hovertext=label,
        stackgroup="one",
        groupnorm="percent",
    ))
fig.update_layout(
    title="WTI Flat Price Contracts — OI Share Over Time (%)",
    xaxis_title="Report Date",
    yaxis_title="% of Total WTI Flat Price OI",
    legend=dict(orientation="h", yanchor="bottom", y=-0.3),
)
fig.show()

## Summary Table — Latest Reporting Date

In [9]:
latest_date = wti["report_date_as_yyyy_mm_dd"].max()
latest = wti[wti["report_date_as_yyyy_mm_dd"] == latest_date].copy()

summary = (
    latest.groupby("cftc_contract_market_code")
    .agg(
        contract_name=("market_and_exchange_names", "first"),
        open_interest=("open_interest_all", "sum"),
    )
    .sort_values("open_interest", ascending=False)
)
summary["pct_of_total"] = (summary["open_interest"] / summary["open_interest"].sum() * 100).round(2)

print(f"Latest report date: {latest_date.date()}")
summary

Latest report date: 2026-03-31


,contract_name,open_interest,pct_of_total
cftc_contract_market_code,,,
067651,WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE,3039583,64.02
067411,"CRUDE OIL, LIGHT SWEET-WTI - ICE FUTURES EUROPE",1184319,24.94
06765A,WTI FINANCIAL CRUDE OIL - NEW YORK MERCANTILE ...,235004,4.95
06765C,CRUDE OIL AVG PRICE OPTIONS - NEW YORK MERCANT...,173578,3.66
067A55,Micro WTI CRUDE OIL FINANCIAL - NEW YORK MERCA...,76474,1.61
06741Q,WTI CRUDE OIL 1ST LINE - ICE FUTURES ENERGY DIV,38890,0.82
